# The Campus Collective — Gemma 4 pipeline, reproduced in Python

**[The Campus Collective (TCC)](https://github.com/CodeWithEugene/The-Campus-Collective)** is an Android app where **Gemma 4 runs fully on-device** (LiteRT-LM via `flutter_gemma`): four specialist agents — 📚 Somo (study), 📜 Karani (bureaucracy), 💸 Hustle (money), 🗓️ Ratiba (planner) — helping University of Embu students in English/Kiswahili/Sheng, offline.

**This notebook is the no-sideload judge path.** It reproduces the app's exact pipeline in Python using our fine-tuned model [`Eugeniuss/gemma-4-tcc-e4b`](https://huggingface.co/Eugeniuss/gemma-4-tcc-e4b) (Gemma 4 E4B + QLoRA, Apache 2.0, ungated):

1. **Trilingual code-switch chat** (what Somo does)
2. **Campus-document → strict JSON extraction** (what Karani does)
3. **Native function calling against the app's real tool schemas + real Safaricom tariff data** (what Hustle does)
4. **RAG with citations over the same UoEm corpus the app ships** (Karani's grounding)

On the phone, the same model runs as a `.litertlm` bundle through LiteRT-LM; here we load the merged HF weights in 4-bit so it fits a free Kaggle T4.

> ⬇️ Try the app itself: [APK on GitHub Releases](https://github.com/CodeWithEugene/The-Campus-Collective/releases/latest/download/app-release.apk)

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

import torch, json, re, urllib.request
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Eugeniuss/gemma-4-tcc-e4b"  # our fine-tune (Apache 2.0, ungated)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
)

def ask(messages, max_new_tokens=256):
    """One chat turn, greedy — same shape as the app's GemmaService.chat()."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Model loaded:", MODEL)

## 1 · Trilingual code-switch (Somo)

The register matters: Kenyan students mix English, Kiswahili and Sheng mid-sentence. This is one of the three skills the fine-tune targets.

In [ ]:
SOMO_SYS = ("You are Somo, The Campus Collective's study helper for University of Embu students. "
            "Reply in the user's language mix (English, Kiswahili or Sheng). Be concise and practical.")

print(ask([
    {"role": "system", "content": SOMO_SYS},
    {"role": "user", "content": "Niaje Somo! Exam ya Data Structures iko next week na sijasoma linked lists. Nisaidie na summary fupi na vile nitakumbuka insertion vs deletion."},
], max_new_tokens=300))

## 2 · Document → strict JSON (Karani)

In the app, Karani receives a **photo** (Gemma 4's on-device vision reads it). The extraction behaviour is identical for text, so judges can verify it here without images: campus document in → the app's `ScanResult`-shaped JSON out, deadlines included (these flow straight into Ratiba's planner).

In [ ]:
FEE_STATEMENT = """UNIVERSITY OF EMBU — FEE STATEMENT 2025/2026
Student: W. Muthoni  Reg: B131/2456/2024  Programme: BSc Computer Science, Year 2 Sem 1
Total billed: KES 68,500 | HELB disbursed: KES 34,000 | Paid (M-Pesa): KES 12,000
Outstanding balance: KES 22,500
NOTE: Students with balances above KES 10,000 must clear or sign a payment plan
by 15 September 2026 to sit end-of-semester examinations."""

raw = ask([
    {"role": "system", "content": "You are Karani, TCC's bureaucracy agent. Extract campus documents into strict JSON."},
    {"role": "user", "content": FEE_STATEMENT + "\n\nRespond with ONLY valid minified JSON: "
     '{"doc_type", "summary_swahili", "fields": {"total_billed", "helb_disbursed", "paid", "balance"}, '
     '"deadlines": [{"title", "date"}], "next_steps": []}'},
], max_new_tokens=350)

m = re.search(r"\{.*\}", raw, re.S)  # same tolerant parse the app uses
result = json.loads(m.group(0))
print(json.dumps(result, indent=2, ensure_ascii=False))

## 3 · Function calling with the app's real tools (Hustle)

The app registers small Dart tools with flat schemas (edge models are the weakest tool-callers, so TCC keeps schemas simple and scopes each agent to its own 2–4 tools). Here we register the same `mpesa_tariff` schema and execute it against **the same Safaricom tariff table the APK bundles** (`data/content/mpesa_tariffs.json`).

In [ ]:
RAW = "https://raw.githubusercontent.com/CodeWithEugene/The-Campus-Collective/main/data/content"
tariffs = json.load(urllib.request.urlopen(f"{RAW}/mpesa_tariffs.json"))

def mpesa_tariff(amount: int, txn_type: str = "send_to_mpesa_user"):
    for band in tariffs[txn_type]:
        if band["min"] <= amount <= band["max"]:
            return {"amount": amount, "txn_type": txn_type, "fee": band["fee"], "currency": "KES"}
    return {"error": "amount out of range"}

TOOL_SYS = ("You are Hustle, TCC's money agent. You have ONE tool:\n"
            'mpesa_tariff(amount: int, txn_type: "send_to_mpesa_user"|"withdraw_agent")\n'
            "When the user asks about M-Pesa costs, respond with ONLY the call as minified JSON: "
            '{"tool": "mpesa_tariff", "args": {...}}. Otherwise answer normally.")

user_q = "Nataka kutuma 2500 bob kwa mathe. Itanikata ngapi na M-Pesa?"
call_raw = ask([{"role": "system", "content": TOOL_SYS}, {"role": "user", "content": user_q}], max_new_tokens=80)
call = json.loads(re.search(r"\{.*\}", call_raw, re.S).group(0))
print("model emitted tool call →", call)

tool_result = mpesa_tariff(**call["args"])
print("tool executed →", tool_result)

print("\nfinal answer →", ask([
    {"role": "system", "content": TOOL_SYS},
    {"role": "user", "content": user_q},
    {"role": "assistant", "content": json.dumps(call)},
    {"role": "user", "content": f"Tool result: {json.dumps(tool_result)}. Now answer the user in their language."},
], max_new_tokens=120))

## 4 · RAG with citations (Karani's grounding)

Fee/HELB facts are never hard-coded: the app retrieves chunks from a corpus of official University of Embu sources and cites them. On the phone this uses on-device embeddings; the retrieval → cite → answer contract is identical here (simple lexical scoring so the demo has no extra dependencies).

In [ ]:
corpus = json.load(urllib.request.urlopen(f"{RAW}/corpus_v1.json"))["chunks"]

def retrieve(query, k=2):
    qw = set(re.findall(r"\w+", query.lower()))
    scored = [(len(qw & set(re.findall(r"\w+", c["text"].lower()))), c) for c in corpus]
    return [c for s, c in sorted(scored, key=lambda x: -x[0])[:k] if s > 0]

q = "When does the academic year run at University of Embu, na hostels zinabook aje?"
chunks = retrieve(q)
ctx = "\n\n".join(f"[{c['id']}] ({c['source']}): {c['text']}" for c in chunks)

print(ask([
    {"role": "system", "content": "You are Karani. Answer ONLY from the provided sources and cite chunk ids in brackets. "
     "If the sources don't cover it, say so and advise verifying with the office."},
    {"role": "user", "content": f"Sources:\n{ctx}\n\nQuestion: {q}"},
], max_new_tokens=250))
print("\nretrieved:", [c["id"] for c in chunks])

## What runs where — and honesty notes

| Piece | In this notebook | In the shipped app |
|---|---|---|
| Model | `Eugeniuss/gemma-4-tcc-e4b` (HF, 4-bit) | Gemma 4 E2B/E4B `.litertlm` via LiteRT-LM / `flutter_gemma`, fully offline |
| Vision | text stand-in (same extraction contract) | Gemma 4 multimodal reads the photo on-device |
| Tools | same schemas, Python executor | same schemas, Dart executors over local SQLite |
| RAG | lexical retrieval, same cite contract | on-device embeddings over the bundled corpus |

**Honesty:** Kiembu support is a hand-collected phrasebook (no public dataset exists — we started one); Sheng is fine-tune-level, not native fluency; sensitive outputs in the app are confidence-gated with "verify with the office" guidance.

**Fine-tune:** Unsloth QLoRA on Gemma 4 E4B, loss 1.79 → 0.57 over 380 steps on ~3.3k examples (programmatic tool/extraction pairs + Gemma 4 26B/31B teacher distillation + public Swahili SFT). Full pipeline: [`gemma_model/`](https://github.com/CodeWithEugene/The-Campus-Collective/tree/main/gemma_model).

*Poa!* 🎉